In [1]:
import carla
client = carla.Client('localhost', 2000)
world = client.get_world()

def find_obj(world, keyword, offset=0):
    env_objs = world.get_environment_objects()
    objs = []

    for env_obj in env_objs:
        if keyword in env_obj.name:
            _env_obj = env_obj
            _env_obj.transform.location.z += offset
            objs.append(_env_obj)

    return objs

spawn_point = find_obj(world, "truck_spawn", 0.5)[0]

truck_bp = world.get_blueprint_library().find("vehicle.daf.dafxf")
trailer_bp = world.get_blueprint_library().find("vehicle.trailer.trailer")

In [2]:
from parking_env_2 import *
env = ParkingLotEnv()

/home/falah/carla/My Code/venv3.10/lib/python3.10/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/falah/carla/My Code/venv3.10/lib/python3.10/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [30]:
env_boundary = find_obj(world, keyword="env_boundary")[0]
truck_parking1 = find_obj(world, keyword="truck_parking1")[0]

In [60]:
bbox = env_boundary.bounding_box
vertices = bbox.get_world_vertices(env_boundary.transform)
n_vertices = []

for vertex in vertices:
    n_vertices.append(np.array([vertex.x, vertex.y]))

In [62]:
parking_location = truck_parking1.transform.location
parking = np.array([parking_location.x, parking_location.y])

In [70]:
distance = max([np.linalg.norm(parking - n_vertices[i]) for i in range(8)])

In [71]:
print(distance)

168.4997176105706


In [3]:
env.reset()

Destroying 0 actors...
Truck Spawned.
Trailer Spawned and Attached.
Spawned 14 actors (vehicles + sensors).
Waiting for radar sensor 9...


({'current_stage': 0,
  'distance_to_target': array([13.549156], dtype=float32),
  'angle_difference': array([89.99979], dtype=float32),
  'jackknife_angle': array([-0.00018311], dtype=float32),
  'phi': array([127.28693], dtype=float32),
  'parallel_distance': array([20.22669], dtype=float32),
  'longitudinal_distance': array([10.779949], dtype=float32),
  'radar_data': array([[4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.]], dtype=float32)},
 {})

In [4]:
env.step([-0.5, 0.05])

({'current_stage': 0,
  'distance_to_target': array([13.548918], dtype=float32),
  'angle_difference': array([90.13208], dtype=float32),
  'jackknife_angle': array([-0.12988281], dtype=float32),
  'phi': array([127.07837], dtype=float32),
  'parallel_distance': array([20.212164], dtype=float32),
  'longitudinal_distance': array([10.76292], dtype=float32),
  'radar_data': array([[4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.],
         [4.]], dtype=float32)},
 -0.002810955047607422,
 False,
 False,
 {})

In [59]:
env._get_observation()

{'current_stage': 0,
 'distance_to_target': array([3.5215504], dtype=float32),
 'angle_difference': array([-9.136307], dtype=float32),
 'jackknife_angle': array([-13.38533], dtype=float32),
 'phi': array([-26.6651], dtype=float32),
 'parallel_distance': array([0.15167823], dtype=float32),
 'longitudinal_distance': array([9.009937], dtype=float32),
 'radar_data': array([[4.      ],
        [4.      ],
        [4.      ],
        [4.      ],
        [4.      ],
        [4.      ],
        [4.      ],
        [4.      ],
        [4.      ],
        [3.219301]], dtype=float32)}

In [ ]:
env.destroy_actors()

In [5]:
world.get_actor(world.get_actors().filter("vehicle.daf*")[0].id).apply_control(carla.VehicleControl(brake=0.5, reverse=False))

In [ ]:
vehicle = world.get_actor(world.get_actors().filter("vehicle.daf*")[0].id)
bounding_box = vehicle.bounding_box

bbox_vertices = bounding_box.get_world_vertices(vehicle.get_transform())

lines = [
    (bbox_vertices[0], bbox_vertices[1]), (bbox_vertices[1], bbox_vertices[2]),
    (bbox_vertices[2], bbox_vertices[3]), (bbox_vertices[3], bbox_vertices[0]),
    (bbox_vertices[4], bbox_vertices[5]), (bbox_vertices[5], bbox_vertices[6]),
    (bbox_vertices[6], bbox_vertices[7]), (bbox_vertices[7], bbox_vertices[4]),
    (bbox_vertices[0], bbox_vertices[4]), (bbox_vertices[1], bbox_vertices[5]),
    (bbox_vertices[2], bbox_vertices[6]), (bbox_vertices[3], bbox_vertices[7])
]

for start_point, end_point in lines:
    world.debug.draw_line(
        start_point,
        end_point,
        thickness=0.05,  # Adjust thickness as needed
        color=carla.Color(r=255, g=0, b=0), # Red color
        life_time=100 # How long the line stays visible (adjust for continuous drawing)
    )

In [5]:
world.try_spawn_actor(trailer_bp, spawn_point.transform)

In [ ]:
import numpy as np

# Get the trailer parking object
parking_point = find_obj(world, 'trailer_parking')[0]
parking_location = parking_point.transform.location

# Get the trailer actor
trailer = world.get_actors().filter("vehicle.trailer*")[0]
trailer_transform = trailer.get_transform()

# Calculate trailer's back end position
trailer_yaw_rad = np.deg2rad(trailer_transform.rotation.yaw)
trailer_backward = np.array([-np.cos(trailer_yaw_rad), -np.sin(trailer_yaw_rad)])
trailer_bb = trailer.bounding_box
trailer_back_end = np.array([trailer_transform.location.x - 5.0, trailer_transform.location.y]) + trailer_backward * trailer_bb.extent.x

# Draw line from trailer back end to parking spot
world.debug.draw_line(
    carla.Location(x=trailer_back_end[0], y=trailer_back_end[1], z=0.5),
    carla.Location(x=parking_location.x, y=parking_location.y, z=0.5),
    thickness=0.1,
    color=carla.Color(r=0, g=255, b=0),
    life_time=50.0
)